# LSUN Churches 256×256

This notebook generates four images per published checkpoint using Colab GPU inference.
Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

- [StyleGAN2 church config-f](https://github.com/NVlabs/stylegan2) uses NVIDIA's PyTorch converter for the original TensorFlow weights.
- [DDPM EMA church 256](https://huggingface.co/google/ddpm-ema-church-256) runs the DDPM checkpoint with a 50-step DDIM sampler.
- [CompVis LDM-KL-8 churches](https://github.com/CompVis/latent-diffusion#pretrained-ldms) also uses 50 DDIM steps and decodes the latent to RGB.

The checkpoints generate unconditional samples. Shared seeds make each model's samples repeatable within the same environment. Latency uses batch size one, CUDA synchronization, and a warmup sample. The timing includes latent decoding and excludes setup, downloads, and PNG writes.

Run the last cell to view the grid. Download `churches_pretrained_results.zip` from the Colab Files pane to retain the originals, checkpoint hashes, timings, and any errors. The examples illustrate these three trained systems under these settings.


In [ ]:
"""Download the published LSUN Churches checkpoints in Colab."""

import json
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import torch

ROOT = Path('/content/churches_pretrained')
WEIGHTS = ROOT / 'weights'
RESULTS = ROOT / 'results'
WEIGHTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

STYLEGAN_URL = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2/networks/stylegan2-church-config-f.pkl'
LDM_URL = 'https://ommer-lab.com/files/latent-diffusion/lsun_churches.zip'
DDPM_REPO = 'https://huggingface.co/google/ddpm-ema-church-256'
DDPM_FILES = ['config.json', 'scheduler_config.json', 'diffusion_pytorch_model.bin']


def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        partial = path.with_suffix(path.suffix + '.part')
        subprocess.run(['curl', '--fail', '--location', '--silent', '--show-error',
                        '--retry', '1', '--retry-max-time', '60', '--connect-timeout', '15',
                        '--max-time', '900', '--output', str(partial), url], check=True)
        if partial.stat().st_size == 0:
            raise ValueError(f'Empty response from {url}')
        partial.replace(path)
    return path.stat().st_size


def fetch_group(items):
    results = []
    for url, path in items:
        try:
            size = download(url, path)
        except Exception as error:
            raise RuntimeError(f'{url}: {error}') from error
        results.append({'url': url, 'file': str(path.relative_to(WEIGHTS)), 'bytes': size})
    return results


if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before running this notebook.')
print('GPU:', torch.cuda.get_device_name(), 'PyTorch:', torch.__version__, flush=True)
groups = {
    'StyleGAN2': [(STYLEGAN_URL, WEIGHTS / 'stylegan2-church-config-f.pkl')],
    'DDPM': [(f'{DDPM_REPO}/resolve/main/{name}', WEIGHTS / 'ddpm' / name)
             for name in DDPM_FILES],
    'LDM': [(LDM_URL, WEIGHTS / 'lsun_churches.zip')],
}
downloads = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    jobs = {pool.submit(fetch_group, items): name for name, items in groups.items()}
    for job in as_completed(jobs):
        name = jobs[job]
        try:
            downloads[name] = {'files': job.result()}
            print(name, 'checkpoint downloaded:', downloads[name]['files'], flush=True)
        except Exception as error:
            downloads[name] = {'error': str(error)}
            print('CHECKPOINT LINK ERROR', name, str(error), flush=True)
        (RESULTS / 'downloads.json').write_text(json.dumps(downloads, indent=2))
print('Download checks finished.', flush=True)


In [ ]:
"""Sample from the published checkpoints downloaded by churches_setup.py."""

import csv
import gc
import hashlib
import shutil
import statistics
import time
import zipfile

import numpy as np
from PIL import Image, ImageDraw

SEEDS = [2026, 2027, 2028, 2029]
DDIM_STEPS = 50
MODELS = ('StyleGAN2', 'DDPM', 'LDM')
RUN_MODELS = globals().pop('ONLY_MODELS', MODELS)


def install(*packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)


def repository(owner, name):
    path = ROOT / name
    if not path.exists():
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1',
                        f'https://github.com/{owner}/{name}.git', str(path)], check=True)
    rev = subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip()
    return path, rev


def rgb_image(tensor):
    array = tensor[0].detach().float().clamp(-1, 1).add(1).mul(127.5).round()
    return Image.fromarray(array.to(torch.uint8).permute(1, 2, 0).cpu().numpy())


def stylegan2():
    source, revision = repository('NVlabs', 'stylegan2-ada-pytorch')
    sys.path.insert(0, str(source))
    import legacy
    from torch_utils.ops import bias_act, upfirdn2d

    # Use NVIDIA's PyTorch reference operations on current Colab runtimes.
    bias_act._init = lambda: False
    upfirdn2d._init = lambda: False
    with (WEIGHTS / 'stylegan2-church-config-f.pkl').open('rb') as file:
        generator = legacy.load_network_pkl(file)['G_ema'].eval().cuda()
    assert (generator.img_resolution, generator.c_dim) == (256, 0)

    def sample(seed):
        latent = np.random.RandomState(seed).randn(1, generator.z_dim).astype('float32')
        return generator(torch.from_numpy(latent).cuda(), None, truncation_psi=1,
                         noise_mode='const', force_fp32=True)

    return sample, {'sampler': 'Direct', 'truncation_psi': 1, 'noise_mode': 'const',
                    'source_revision': revision, 'backend': 'NVIDIA PyTorch legacy converter + reference ops'}


def ddpm():
    install('diffusers==0.35.2')
    from diffusers import DDIMScheduler, UNet2DModel
    from importlib.metadata import version

    folder = WEIGHTS / 'ddpm'
    model, load_info = UNet2DModel.from_pretrained(folder, use_safetensors=False,
                                                  output_loading_info=True)
    if load_info['missing_keys'] or load_info['mismatched_keys']:
        raise ValueError(f'Incomplete DDPM checkpoint: {load_info}')
    model = model.eval().cuda()
    assert model.config.sample_size == 256
    scheduler = DDIMScheduler.from_pretrained(folder)
    scheduler.set_timesteps(DDIM_STEPS, device='cuda')

    def sample(seed):
        rng = torch.Generator(device='cuda').manual_seed(seed)
        image = torch.randn((1, 3, 256, 256), generator=rng, device='cuda')
        for t in scheduler.timesteps:
            noise = model(image, t).sample
            image = scheduler.step(noise, t, image, eta=0).prev_sample
        return image

    return sample, {'sampler': 'DDIM', 'steps': DDIM_STEPS, 'eta': 0,
                    'backend': f'Diffusers {version("diffusers")}'}


def ldm():
    install('omegaconf==2.3.0', 'einops', 'pytorch-lightning')
    from omegaconf import OmegaConf

    source, revision = repository('CompVis', 'latent-diffusion')
    taming, taming_revision = repository('CompVis', 'taming-transformers')
    sys.path[:0] = [str(source), str(taming)]
    module_file = source / 'ldm/models/diffusion/ddpm.py'
    old_import = 'from pytorch_lightning.utilities.distributed import rank_zero_only'
    new_import = 'from pytorch_lightning.utilities.rank_zero import rank_zero_only'
    code = module_file.read_text()
    if old_import in code:
        module_file.write_text(code.replace(old_import, new_import))
    import ldm.models.diffusion.ddim as ddim_module
    from ldm.util import instantiate_from_config

    archive_path = WEIGHTS / 'lsun_churches.zip'
    with zipfile.ZipFile(archive_path) as archive:
        configs = [name for name in archive.namelist() if Path(name).name == 'config.yaml']
        models = [name for name in archive.namelist() if Path(name).name == 'model.ckpt']
        if len(models) != 1 or len(configs) > 1:
            raise ValueError(f'Unexpected checkpoint archive layout in {LDM_URL}')
        if configs:
            config = OmegaConf.create(archive.read(configs[0]).decode())
        else:
            config = OmegaConf.load(source / 'configs/latent-diffusion/lsun_churches-ldm-kl-8.yaml')
        checkpoint = WEIGHTS / 'ldm_churches.ckpt'
        if not checkpoint.exists():
            partial = checkpoint.with_suffix('.part')
            with archive.open(models[0]) as data, partial.open('wb') as file:
                shutil.copyfileobj(data, file, length=4 * 1024 * 1024)
            partial.replace(checkpoint)

    # The authors' checkpoint includes the first-stage autoencoder.
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)['state_dict']
    params = config.model.params
    params.first_stage_config.params.pop('ckpt_path', None)
    params.first_stage_config.params.lossconfig = {'target': 'torch.nn.Identity'}
    params.pop('ckpt_path', None)
    params.use_ema = any(key.startswith('model_ema.') for key in state)
    model = instantiate_from_config(config.model)
    missing, _ = model.load_state_dict(state, strict=False)
    required = [key for key in missing if key.startswith(('model.', 'model_ema.', 'first_stage_model.'))
                or key == 'scale_factor']
    if required:
        raise ValueError(f'Incomplete LDM checkpoint: {required[:15]}')
    del state
    if model.use_ema:
        model.model_ema.copy_to(model.model)
        del model.model_ema
    model = model.eval().cuda()
    assert model.cond_stage_model is None and model.channels == 4 and model.image_size == 32
    sampler = ddim_module.DDIMSampler(model)
    ddim_module.tqdm = lambda iterable, **kwargs: iterable
    sampler.make_schedule(ddim_num_steps=DDIM_STEPS, ddim_eta=0, verbose=False)

    def sample(seed):
        rng = torch.Generator(device='cuda').manual_seed(seed)
        noise = torch.randn((1, 4, 32, 32), generator=rng, device='cuda')
        latents, _ = sampler.ddim_sampling(None, noise.shape, x_T=noise)
        return model.decode_first_stage(latents)

    return sample, {'sampler': 'DDIM', 'steps': DDIM_STEPS, 'eta': 0, 'ema_buffers': model.use_ema,
                    'source_revision': revision, 'taming_revision': taming_revision,
                    'backend': 'CompVis latent-diffusion; updated Lightning import'}


def run_model(name, loader):
    print('Loading', name, flush=True)
    start = time.perf_counter()
    sample, details = loader()
    torch.cuda.synchronize()
    setup_seconds = time.perf_counter() - start
    model_dir = RESULTS / name
    model_dir.mkdir(exist_ok=True)
    times = []
    with torch.inference_mode():
        warmup = sample(2025)
        del warmup
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        for seed in SEEDS:
            start = time.perf_counter()
            output = sample(seed)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            assert tuple(output.shape) == (1, 3, 256, 256) and torch.isfinite(output).all()
            rgb_image(output).save(model_dir / f'{seed}.png')
            del output
            times.append(elapsed)
            print(f'{name} seed {seed}: {elapsed:.3f}s, 256x256', flush=True)
    details.update(seeds=SEEDS, seconds_per_seed=times, median_seconds=statistics.median(times),
                   setup_seconds=setup_seconds, peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated())
    for record in downloads[name]['files']:
        with (WEIGHTS / record['file']).open('rb') as file:
            record['sha256'] = hashlib.file_digest(file, 'sha256').hexdigest()
    details['checkpoint_files'] = downloads[name]['files']
    (model_dir / 'result.json').write_text(json.dumps(details, indent=2))


loaders = {'StyleGAN2': stylegan2, 'DDPM': ddpm, 'LDM': ldm}
errors_path = RESULTS / 'errors.json'
errors = json.loads(errors_path.read_text()) if errors_path.exists() else {}
for name in RUN_MODELS:
    if 'error' in downloads[name]:
        errors[name] = downloads[name]['error']
        continue
    try:
        run_model(name, loaders[name])
        errors.pop(name, None)
    except Exception:
        import traceback
        errors[name] = traceback.format_exc()
        print(f'INFERENCE ERROR {name}\n{errors[name]}', flush=True)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
errors_path.write_text(json.dumps(errors, indent=2))

records = {name: json.loads((RESULTS / name / 'result.json').read_text())
           for name in MODELS if (RESULTS / name / 'result.json').is_file() and name not in errors}
if records:
    report = Image.new('RGB', (len(SEEDS) * 266 + 12, 76 + len(records) * 290), '#f5f5f2')
    draw = ImageDraw.Draw(report)
    draw.text((12, 10), 'LSUN Churches 256 x 256 | Published checkpoints | Colab inference', fill='#222222')
    draw.text((12, 30), 'One independent sample per seed and model | Batch 1 | All measured images', fill='#222222')
    for column, seed in enumerate(SEEDS):
        draw.text((12 + column * 266, 55), f'Seed {seed}', fill='#222222')
    for row, (name, details) in enumerate(records.items()):
        top = 76 + row * 290
        sampler = details['sampler'] + (f' {details["steps"]}' if 'steps' in details else '')
        draw.text((12, top), f'{name} | {sampler} | median {details["median_seconds"]:.3f} s/image', fill='#222222')
        for column, seed in enumerate(SEEDS):
            with Image.open(RESULTS / name / f'{seed}.png') as sample_image:
                report.paste(sample_image, (12 + column * 266, top + 20))
    report.save(RESULTS / 'comparison.png')

with (RESULTS / 'summary.csv').open('w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['model', 'sampler', 'steps', 'samples', 'median_seconds_per_image'])
    for name, info in records.items():
        writer.writerow([name, info['sampler'], info.get('steps', 1), len(info['seeds']), info['median_seconds']])
        print(name, info['sampler'], f'{info["median_seconds"]:.3f} median seconds/image', flush=True)
(RESULTS / 'run.json').write_text(json.dumps({'dataset': 'LSUN Churches', 'resolution': [256, 256],
    'conditioning': 'unconditional', 'batch_size': 1, 'warmup_seed': 2025, 'trained_here': False,
    'device': torch.cuda.get_device_name(), 'torch': torch.__version__, 'precision': 'float32',
    'models_completed': list(records), 'errors': list(errors)}, indent=2))
export = shutil.make_archive('/content/churches_pretrained_results', 'zip', RESULTS)
print('RESULTS ZIP:', export, Path(export).stat().st_size, 'bytes', flush=True)


In [ ]:
from IPython.display import display
if (RESULTS / 'comparison.png').exists():
    display(Image.open(RESULTS / 'comparison.png'))
print('Download from the Files pane:', export)


In [ ]:
"""Colab cell: inspect one LSUN Churches DDPM sample and its intermediate states."""

import json
import shutil
from pathlib import Path

import torch
from diffusers import DDIMScheduler, UNet2DModel
from PIL import Image, ImageDraw, ImageFont


def export_ddpm_stages(weights, destination):
    seed = 2026
    reverse_steps = 50
    reverse_marks = [0, 10, 20, 30, 40, 50]
    forward_marks = [0, 50, 150, 300, 600, 1000]
    inspect_step = 25
    device = 'cuda'

    for folder in ('reverse', 'forward', 'one_step'):
        (destination / folder).mkdir(parents=True, exist_ok=True)
    scheduler = DDIMScheduler.from_pretrained(weights)
    scheduler.set_timesteps(reverse_steps, device=device)
    assert scheduler.config.prediction_type == 'epsilon'
    assert scheduler.config.num_train_timesteps == forward_marks[-1]
    model = UNet2DModel.from_pretrained(weights, use_safetensors=False).eval().to(device)

    def save_frame(relative, tensor, limit=1.0):
        frame = tensor[0].detach().float().div(limit).clamp(-1, 1)
        pixels = frame.add(1).mul(127.5).round().to(torch.uint8).permute(1, 2, 0)
        image = Image.fromarray(pixels.cpu().numpy())
        assert image.size == (256, 256)
        image.save(destination / relative)
        return image

    reverse_images = []
    forward_images = []
    detail_images = []
    with torch.inference_mode():
        rng = torch.Generator(device=device).manual_seed(seed)
        sample = torch.randn((1, 3, 256, 256), generator=rng, device=device)
        reverse_images.append(('0 / 50 | Gaussian noise', save_frame('reverse/step_000.png', sample)))
        for step, timestep in enumerate(scheduler.timesteps, 1):
            predicted_noise = model(sample, timestep).sample
            update = scheduler.step(predicted_noise, timestep, sample, eta=0)
            if step == inspect_step:
                inspected_t = int(timestep)
                detail_images = [
                    ('1. Noisy input x_t', save_frame('one_step/1_input.png', sample)),
                    ('2. Noise predicted by U-Net', save_frame('one_step/2_noise.png', predicted_noise, limit=2.5)),
                    ('3. Clean-image estimate', save_frame('one_step/3_clean_estimate.png', update.pred_original_sample)),
                    ('4. Updated sample', save_frame('one_step/4_next_state.png', update.prev_sample)),
                ]
            sample = update.prev_sample
            frame = save_frame(f'reverse/step_{step:03d}.png', sample)
            if step in reverse_marks:
                label = f'{step} / 50' + (' | Final image' if step == reverse_steps else '')
                reverse_images.append((label, frame))
                print('Saved reverse step', step, '/', reverse_steps, flush=True)

        # A separate Markov forward path starts from the generated clean image.
        forward_rng = torch.Generator(device=device).manual_seed(seed + 1)
        sample = sample.clamp(-1, 1)
        betas = scheduler.betas.to(device=device, dtype=sample.dtype)
        forward_images.append(('0 / 1000 | Starting image', save_frame('forward/step_0000.png', sample)))
        for step, beta in enumerate(betas, 1):
            noise = torch.randn(sample.shape, generator=forward_rng, device=device)
            sample = (1 - beta).sqrt() * sample + beta.sqrt() * noise
            if step in forward_marks:
                frame = save_frame(f'forward/step_{step:04d}.png', sample)
                forward_images.append((f'{step} / 1000', frame))

    def sheet(title, subtitle, rows, filename):
        margin, spacing, row_height = 16, 12, 312
        columns = max(len(frames) for _, frames in rows)
        width = 2 * margin + columns * 256 + (columns - 1) * spacing
        canvas = Image.new('RGB', (width, 74 + len(rows) * row_height), '#f5f5f2')
        draw = ImageDraw.Draw(canvas)
        title_font = ImageFont.load_default(size=25)
        label_font = ImageFont.load_default(size=16)
        draw.text((margin, 10), title, fill='#161616', font=title_font)
        draw.text((margin, 44), subtitle, fill='#454545', font=label_font)
        for index, (heading, frames) in enumerate(rows):
            top = 74 + row_height * index
            draw.text((margin, top), heading, fill='#161616', font=label_font)
            for column, (label, image) in enumerate(frames):
                left = margin + (256 + spacing) * column
                canvas.paste(image, (left, top + 25))
                draw.text((left, top + 285), label, fill='#161616', font=label_font)
        canvas.save(destination / filename)

    sheet('DDPM | One LSUN Churches image (256 x 256)',
          'Seed 2026 | The two rows show separate paths with opposite directions.',
          [('Reverse generation | Gaussian noise to image | DDIM, 50 steps', reverse_images),
           ('Forward diffusion | Start from that image and add noise | 1,000 steps', forward_images)],
          'trajectory.png')
    sheet(f'Inside reverse step {inspect_step} / {reverse_steps}',
          'The U-Net predicts noise. DDIM uses the prediction to estimate the clean image and update the sample.',
          [(f'Scheduler input t = {inspected_t} | Predicted noise displayed over [-2.5, 2.5]', detail_images)],
          'one_step.png')

    (destination / 'stages.json').write_text(json.dumps({
        'checkpoint': 'google/ddpm-ema-church-256', 'seed': seed,
        'reverse_sampler': 'DDIM', 'eta': 0, 'reverse_steps': reverse_steps,
        'reverse_frame_steps': reverse_marks, 'scheduler_timesteps': scheduler.timesteps.tolist(),
        'inspected_step': inspect_step, 'inspected_timestep': inspected_t,
        'forward_steps': len(betas), 'forward_frame_steps': forward_marks, 'forward_seed': seed + 1,
        'gpu': torch.cuda.get_device_name(), 'torch': torch.__version__,
    }, indent=2))
    archive = shutil.make_archive(str(destination), 'zip', destination)
    print('Saved 51 reverse frames, 6 forward frames and 4 panels for one step.', flush=True)
    print('Download from Colab Files:', archive, flush=True)


export_ddpm_stages(Path('/content/churches_pretrained/weights/ddpm'), Path('/content/ddpm_churches_stages'))


## Five ImageNet labels at 256×256

Run the next two cells on a Colab GPU. They use the released OpenAI class-conditional ADM checkpoint and CompVis LDM ImageNet checkpoint (cin256-v2, VQ-f4). Both use 50 DDIM steps and the same seed and class IDs. LDM uses classifier-free guidance at scale 3.0, as in the CompVis demo.

The report shows reverse denoising, predicted noise inside step 25, and a separate forward-noising pass from each generated sample. LDM stages are decoded for display. Its predicted noise is shown as a latent heatmap.

Download `imagenet_diffusion_stages_final.zip` from the Colab Files pane.


In [ ]:
"""Colab setup for five shared ImageNet classes at 256 x 256."""

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import subprocess

IMAGENET_ROOT = Path('/content/imagenet_diffusion_stages')
IMAGENET_ROOT.mkdir(parents=True, exist_ok=True)
IMAGENET_WEIGHTS = {
    'ADM': ('https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion.pt',
            IMAGENET_ROOT / 'adm_256.pt'),
    'LDM': ('https://ommer-lab.com/files/latent-diffusion/nitro/cin/model.ckpt',
            IMAGENET_ROOT / 'cin256-v2.ckpt'),
}
IMAGENET_CLASSES = {
    207: 'Golden retriever',
    281: 'Tabby cat',
    779: 'School bus',
    323: 'Monarch butterfly',
    980: 'Volcano',
}


def fetch_imagenet_weights(url, path):
    if not path.is_file():
        partial = path.with_suffix(path.suffix + '.part')
        subprocess.run(['curl', '-fLsS', '--retry', '1', '--connect-timeout', '20',
                        '--max-time', '900', '-o', str(partial), url], check=True)
        if partial.stat().st_size < 100_000_000:
            raise ValueError(f'Unexpectedly small checkpoint from {url}')
        partial.replace(path)
    return path.stat().st_size


IMAGENET_READY = {}
with ThreadPoolExecutor(max_workers=2) as pool:
    jobs = {pool.submit(fetch_imagenet_weights, *spec): name for name, spec in IMAGENET_WEIGHTS.items()}
    for job in as_completed(jobs):
        name = jobs[job]
        try:
            print(f'{name} checkpoint ready: {job.result():,} bytes', flush=True)
            IMAGENET_READY[name] = True
        except Exception as error:
            IMAGENET_READY[name] = False
            print(f'CHECKPOINT DOWNLOAD ERROR {name}: {IMAGENET_WEIGHTS[name][0]}\n{error}', flush=True)


In [ ]:
"""Capture five paired ImageNet generation traces in the existing Colab notebook."""

import gc
import json
import shutil
import subprocess
import sys
import traceback
from pathlib import Path

import torch
from matplotlib import colormaps
from PIL import Image, ImageDraw, ImageFont

IMAGENET_SEED = 2026
REVERSE_STEPS = 50
REVERSE_MARKS = (0, 10, 20, 30, 40, 50)
FORWARD_MARKS = (0, 50, 150, 300, 600, 1000)
INSPECT_STEP = 25
DEVICE = 'cuda'
RESULTS_IM = IMAGENET_ROOT / 'results_cin256_v2'
RESULTS_IM.mkdir(exist_ok=True)
for class_id in IMAGENET_CLASSES:
    previous_adm = IMAGENET_ROOT / 'results' / str(class_id) / 'ADM'
    current_adm = RESULTS_IM / str(class_id) / 'ADM'
    if (previous_adm / 'trace.json').is_file() and not current_adm.exists():
        shutil.copytree(previous_adm, current_adm)


def model_repository(owner, name):
    path = Path('/content/churches_pretrained') / name
    if not (path / '.git').is_dir():
        path = IMAGENET_ROOT / name
        if not (path / '.git').is_dir():
            subprocess.run(['git', 'clone', '--quiet', '--depth', '1',
                            f'https://github.com/{owner}/{name}.git', str(path)], check=True)
    sys.path.insert(0, str(path))
    return path


def load_adm():
    model_repository('openai', 'guided-diffusion')
    from guided_diffusion.gaussian_diffusion import get_named_beta_schedule
    from guided_diffusion.script_util import create_model_and_diffusion, model_and_diffusion_defaults

    config = model_and_diffusion_defaults()
    config.update(image_size=256, attention_resolutions='32,16,8', class_cond=True,
                  diffusion_steps=1000, learn_sigma=True, noise_schedule='linear',
                  num_channels=256, num_head_channels=64, num_res_blocks=2,
                  resblock_updown=True, use_fp16=True, use_scale_shift_norm=True,
                  timestep_respacing='ddim50')
    model, diffusion = create_model_and_diffusion(**config)
    state = torch.load(IMAGENET_WEIGHTS['ADM'][1], map_location='cpu', weights_only=True)
    model.load_state_dict(state)
    del state
    model.convert_to_fp16()
    model.eval().to(DEVICE)

    def step(sample, index, label):
        captured_noise = None

        def predict(value, timesteps, **kwargs):
            nonlocal captured_noise
            result = model(value, timesteps, **kwargs)
            captured_noise = result[:, :3]
            return result

        position = REVERSE_STEPS - index
        timestep = torch.tensor([position], device=DEVICE)
        update = diffusion.ddim_sample(predict, sample, timestep,
                                       model_kwargs={'y': label}, eta=0)
        return update['sample'], update['pred_xstart'], captured_noise, diffusion.timestep_map[position]

    betas = torch.as_tensor(get_named_beta_schedule('linear', 1000), device=DEVICE, dtype=torch.float32)
    return step, lambda state: state, (1, 3, 256, 256), betas


def load_imagenet_ldm():
    source = model_repository('CompVis', 'latent-diffusion')
    model_repository('CompVis', 'taming-transformers')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'omegaconf==2.3.0', 'einops', 'pytorch-lightning', 'kornia',
                    'git+https://github.com/openai/CLIP.git'], check=True)
    from omegaconf import OmegaConf

    module = source / 'ldm/models/diffusion/ddpm.py'
    old_import = 'from pytorch_lightning.utilities.distributed import rank_zero_only'
    content = module.read_text()
    if old_import in content:
        module.write_text(content.replace(old_import, 'from pytorch_lightning.utilities.rank_zero import rank_zero_only'))
    from ldm.models.diffusion.ddim import DDIMSampler
    from ldm.util import instantiate_from_config

    checkpoint = IMAGENET_WEIGHTS['LDM'][1]
    config = OmegaConf.load(source / 'configs/latent-diffusion/cin256-v2.yaml')
    try:
        state = torch.load(checkpoint, map_location='cpu', weights_only=False, mmap=True)['state_dict']
    except RuntimeError as error:
        if 'mmap' not in str(error):
            raise
        state = torch.load(checkpoint, map_location='cpu', weights_only=False)['state_dict']
    params = config.model.params
    params.pop('ckpt_path', None)
    params.first_stage_config.params.pop('ckpt_path', None)
    params.first_stage_config.params.lossconfig = {'target': 'torch.nn.Identity'}
    params.cond_stage_config.params.n_classes = state['cond_stage_model.embedding.weight'].shape[0]
    model = instantiate_from_config(config.model)
    missing, _ = model.load_state_dict(state, strict=False)
    required = [key for key in missing if key.startswith(('model.', 'first_stage_model.', 'cond_stage_model.'))
                or key == 'scale_factor']
    if required:
        raise ValueError(f'Incomplete LDM weights: {required[:10]}')
    del state
    model.eval().to(DEVICE)
    assert model.channels == 3 and model.image_size == 64 and model.parameterization == 'eps'
    sampler = DDIMSampler(model)
    sampler.make_schedule(ddim_num_steps=REVERSE_STEPS, ddim_eta=0, verbose=False)

    class RecordNoise:
        def modify_score(self, model, predicted, sample, timestep, condition):
            self.predicted = predicted
            return predicted

    def step(sample, index, conditioning):
        condition, unconditional = conditioning
        position = REVERSE_STEPS - index
        original_timestep = int(sampler.ddim_timesteps[position])
        timestep = torch.tensor([original_timestep], device=DEVICE)
        record = RecordNoise()
        update, estimate = sampler.p_sample_ddim(sample, condition, timestep, index=position,
                                                  score_corrector=record, corrector_kwargs={},
                                                  unconditional_guidance_scale=3.0,
                                                  unconditional_conditioning=unconditional)
        return update, estimate, record.predicted, original_timestep

    def condition_for(label):
        condition = model.get_learned_conditioning({model.cond_stage_key: label})
        unconditional = model.get_learned_conditioning({model.cond_stage_key: torch.full_like(label, 1000)})
        return condition, unconditional

    return step, model.decode_first_stage, (1, 3, 64, 64), model.betas, condition_for


def frame(tensor, limit=1.0):
    assert torch.isfinite(tensor).all()
    pixel = tensor[0].detach().float().div(limit).clamp(-1, 1).add(1).mul(127.5).round()
    return Image.fromarray(pixel.to(torch.uint8).permute(1, 2, 0).cpu().numpy())


def latent_noise_map(tensor):
    rms = tensor[0].float().square().mean(dim=0).sqrt().cpu().numpy()
    colors = colormaps['magma']((rms / 2.5).clip(0, 1), bytes=True)[..., :3]
    return Image.fromarray(colors).resize((256, 256), Image.Resampling.NEAREST)


def sheet(path, title, subtitle, rows, tile=176):
    margin, gap, row_height = 14, 10, tile + 50
    columns = max(len(frames) for _, frames in rows)
    canvas = Image.new('RGB', (2 * margin + columns * tile + (columns - 1) * gap,
                              68 + row_height * len(rows)), '#f5f5f2')
    draw = ImageDraw.Draw(canvas)
    heading_font = ImageFont.load_default(size=22)
    font = ImageFont.load_default(size=14)
    draw.text((margin, 9), title, fill='#171717', font=heading_font)
    draw.text((margin, 38), subtitle, fill='#454545', font=font)
    for row, (heading, frames) in enumerate(rows):
        top = 68 + row_height * row
        draw.text((margin, top), heading, fill='#171717', font=font)
        for col, (caption, image) in enumerate(frames):
            left = margin + (tile + gap) * col
            canvas.paste(image.resize((tile, tile), Image.Resampling.LANCZOS), (left, top + 20))
            draw.text((left, top + tile + 25), caption, fill='#171717', font=font)
    canvas.save(path)


def capture_label(name, class_id, label_name, step, decode, shape, betas, condition_for):
    folder = RESULTS_IM / str(class_id) / name
    folder.mkdir(parents=True, exist_ok=True)
    rng = torch.Generator(device=DEVICE).manual_seed(IMAGENET_SEED)
    forward_rng = torch.Generator(device=DEVICE).manual_seed(IMAGENET_SEED + 1)
    label = torch.tensor([class_id], device=DEVICE)
    condition = condition_for(label)
    state = torch.randn(shape, generator=rng, device=DEVICE)
    reverse = [('0 / 50', frame(decode(state)))]
    for index in range(1, REVERSE_STEPS + 1):
        updated, estimate, noise, timestep = step(state, index, condition)
        if index == INSPECT_STEP:
            inspect_t = timestep
            noise_image = frame(noise, 2.5) if name == 'ADM' else latent_noise_map(noise)
            detail = [('Noisy input', frame(decode(state))), ('Predicted noise', noise_image),
                      ('Clean estimate', frame(decode(estimate))), ('Updated state', frame(decode(updated)))]
        state = updated
        if index in REVERSE_MARKS:
            reverse.append((f'{index} / 50', frame(decode(state))))

    final_image = reverse[-1][1]
    final_image.save(folder / 'final.png')
    forward = [('0 / 1000', final_image)]
    if name == 'ADM':
        state = state.clamp(-1, 1)
    for index, beta in enumerate(betas, 1):
        noise = torch.randn(state.shape, generator=forward_rng, device=DEVICE)
        state = (1 - beta).sqrt() * state + beta.sqrt() * noise
        if index in FORWARD_MARKS:
            forward.append((f'{index} / 1000', frame(decode(state))))

    for kind, frames in [('reverse', reverse), ('detail', detail), ('forward', forward)]:
        for index, (_, image) in enumerate(frames):
            image.save(folder / f'{kind}_{index:02d}.png')
    (folder / 'trace.json').write_text(json.dumps({'label': label_name, 'class_id': class_id,
        'model': name, 'checkpoint': IMAGENET_WEIGHTS[name][0], 'seed': IMAGENET_SEED, 'eta': 0, 'reverse_steps': REVERSE_STEPS,
        'reverse_marks': REVERSE_MARKS, 'forward_marks': FORWARD_MARKS,
        'inspected_step': INSPECT_STEP, 'inspected_timestep': inspect_t}, indent=2))
    print(f'{name} | {label_name}: reverse, predicted noise and forward frames saved', flush=True)


@torch.inference_mode()
def run_imagenet_model(name, labels):
    print(f'Loading {name}', flush=True)
    if name == 'ADM':
        step, decode, shape, betas = load_adm()
        condition_for = lambda label: label
    else:
        step, decode, shape, betas, condition_for = load_imagenet_ldm()
    for class_id, label in labels.items():
        capture_label(name, class_id, label, step, decode, shape, betas, condition_for)


def trace_ready(folder, name):
    trace = folder / 'trace.json'
    if not trace.is_file():
        return False
    metadata = json.loads(trace.read_text())
    if name == 'ADM' and 'checkpoint' not in metadata:
        return True  # Reuse traces from the unchanged ADM setup.
    return metadata.get('checkpoint') == IMAGENET_WEIGHTS[name][0]


def save_imagenet_reports():
    final_rows = []
    completed = []
    for name in IMAGENET_WEIGHTS:
        finals = [(label, Image.open(RESULTS_IM / str(class_id) / name / 'final.png'))
                  for class_id, label in IMAGENET_CLASSES.items()
                  if trace_ready(RESULTS_IM / str(class_id) / name, name)]
        if finals:
            final_rows.append((name, finals))
    for class_id, label in IMAGENET_CLASSES.items():
        folder = RESULTS_IM / str(class_id)
        trajectory_rows, detail_rows = [], []
        for name in IMAGENET_WEIGHTS:
            model_folder = folder / name
            if not trace_ready(model_folder, name):
                continue
            completed.append({'model': name, 'label': label, 'class_id': class_id})
            domain = 'pixels' if name == 'ADM' else 'decoded latents'
            for kind, marks, phase in [('reverse', REVERSE_MARKS, 'denoising'),
                                       ('forward', FORWARD_MARKS, 'adding noise')]:
                images = [(str(mark), Image.open(model_folder / f'{kind}_{index:02d}.png'))
                          for index, mark in enumerate(marks)]
                trajectory_rows.append((f'{name} | {phase} | {domain}', images))
            noise_label = 'Predicted noise' if name == 'ADM' else 'Guided latent noise RMS'
            images = [(caption, Image.open(model_folder / f'detail_{index:02d}.png'))
                      for index, caption in enumerate(('Noisy input', noise_label, 'Clean estimate', 'Updated state'))]
            detail_rows.append((name, images))
        if trajectory_rows:
            sheet(folder / 'trajectory.png', f'{label} | ImageNet {class_id}',
                  'DDIM 50 steps. Forward noising starts from each model\'s final sample.', trajectory_rows)
            sheet(folder / 'one_step.png', f'{label} | Inside DDIM step {INSPECT_STEP}',
                  'ADM noise is shown as RGB. LDM heatmap is channel RMS with display range 0 to 2.5.', detail_rows, tile=216)
    if final_rows:
        sheet(RESULTS_IM / 'comparison.png', 'Five ImageNet labels | ADM and LDM',
              '256 x 256, seed 2026. ADM class conditioning. LDM classifier-free guidance 3.0.', final_rows, tile=210)
    (RESULTS_IM / 'run.json').write_text(json.dumps({'completed': completed, 'gpu': torch.cuda.get_device_name(),
        'torch': torch.__version__, 'checkpoints': {name: spec[0] for name, spec in IMAGENET_WEIGHTS.items()},
        'guidance': {'ADM': 'class conditioning', 'LDM': 'classifier-free guidance 3.0'}, 'trained_here': False}, indent=2))
    archive = shutil.make_archive('/content/imagenet_diffusion_stages_final', 'zip', RESULTS_IM)
    print(f'Completed {len(completed)} / 10 traces | {archive}', flush=True)
    if final_rows:
        from IPython.display import display
        display(Image.open(RESULTS_IM / 'comparison.png'))


errors = {}
for name in IMAGENET_WEIGHTS:
    if not IMAGENET_READY.get(name):
        errors[name] = f'Checkpoint download failed: {IMAGENET_WEIGHTS[name][0]}'
        continue
    remaining = {class_id: label for class_id, label in IMAGENET_CLASSES.items()
                 if not trace_ready(RESULTS_IM / str(class_id) / name, name)}
    try:
        if remaining:
            run_imagenet_model(name, remaining)
    except Exception:
        errors[name] = traceback.format_exc()
        print(errors[name], flush=True)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
(RESULTS_IM / 'errors.json').write_text(json.dumps(errors, indent=2))
save_imagenet_reports()


In [ ]:
#@title View denoising and forward noising for one ImageNet label
from IPython.display import display

selected_label = 'Golden retriever' #@param ['Golden retriever', 'Tabby cat', 'School bus', 'Monarch butterfly', 'Volcano']
selected_id = next(i for i, label in IMAGENET_CLASSES.items() if label == selected_label)
selected_report = RESULTS_IM / str(selected_id)
for report in ('trajectory.png', 'one_step.png'):
    display(Image.open(selected_report / report))
